In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.metrics import mean_squared_error

In [2]:
data = np.load("../data/processed/windows_no_leak/window_60.npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

print(X_train.shape)

(11841, 600)


In [3]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [4]:
class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [12]:
class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()


        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.model(x)

In [13]:
input_dim = X_train.shape[1]

model = ANN(input_dim)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [16]:
# epochs = 20
epochs=50
batch_size = 128

for epoch in range(epochs):
    total_loss = 0

    perm = torch.randperm(X_train.size(0))

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]

        xb = X_train[idx]
        yb = y_train[idx]

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    avg_loss = total_loss / (X_train.size(0) // batch_size)
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 19.8533
Epoch 2, Loss: 14.7039
Epoch 3, Loss: 10.9468
Epoch 4, Loss: 22.0559
Epoch 5, Loss: 12.8201
Epoch 6, Loss: 8.1624
Epoch 7, Loss: 9.2976
Epoch 8, Loss: 7.2271
Epoch 9, Loss: 6.9255
Epoch 10, Loss: 13.4384
Epoch 11, Loss: 10.1181
Epoch 12, Loss: 6.2702
Epoch 13, Loss: 7.2386
Epoch 14, Loss: 6.3222
Epoch 15, Loss: 5.4679
Epoch 16, Loss: 3.9187
Epoch 17, Loss: 2.9629
Epoch 18, Loss: 3.3729
Epoch 19, Loss: 2.1886
Epoch 20, Loss: 1.9973
Epoch 21, Loss: 2.5823
Epoch 22, Loss: 1.7071
Epoch 23, Loss: 2.9235
Epoch 24, Loss: 1.9491
Epoch 25, Loss: 1.3992
Epoch 26, Loss: 1.0649
Epoch 27, Loss: 1.7457
Epoch 28, Loss: 1.0325
Epoch 29, Loss: 1.9421
Epoch 30, Loss: 1.5121
Epoch 31, Loss: 1.0551
Epoch 32, Loss: 1.6519
Epoch 33, Loss: 1.0940
Epoch 34, Loss: 1.7598
Epoch 35, Loss: 1.0988
Epoch 36, Loss: 2.6664
Epoch 37, Loss: 1.2069
Epoch 38, Loss: 0.8533
Epoch 39, Loss: 2.0545
Epoch 40, Loss: 1.9199
Epoch 41, Loss: 1.6633
Epoch 42, Loss: 1.4556
Epoch 43, Loss: 0.9165
Epoch 44, Los

In [17]:
model.eval()

with torch.no_grad():
    pred = model(X_test).numpy()

rmse = np.sqrt(mean_squared_error(y_test.numpy(), pred))

print("ANN RMSE:", rmse)

ANN RMSE: 11.656948365263942
